# E-commerce Business Analysis

This notebook presents core business performance, monthly trends, category analysis, review distribution, and fulfillment insights for the Olist e-commerce dataset.

## Project Background

- Data source: Olist Brazilian E-commerce Dataset
- Analysis scope: delivered-order business performance, category contribution, review quality, and fulfillment experience
- Data source table for notebook analysis: `order_level_dataset`

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11


In [ ]:
def load_env_file(env_path: str = '../.env') -> None:
    env_file = Path(env_path)
    if not env_file.exists():
        return

    for line in env_file.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip())

load_env_file()

DB_HOST = os.getenv('DB_HOST', '127.0.0.1')
DB_PORT = os.getenv('DB_PORT', '3306')
DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD', '')
DB_NAME = os.getenv('DB_NAME', 'ecommerce_gmv_customer_analysis')

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4'
)

PROJECT_ROOT = Path('../').resolve()
DASHBOARD_DIR = PROJECT_ROOT / 'dashboards'
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)

print('Connected to database:', DB_NAME)
print('Dashboard directory:', DASHBOARD_DIR)


## Overall KPI Summary

In [ ]:
overall_kpi_sql = """
SELECT
    ROUND(SUM(gmv), 2) AS total_gmv,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT customer_unique_id) AS total_buyers,
    ROUND(SUM(gmv) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS overall_aov
FROM order_level_dataset
WHERE order_status = 'delivered';
"""

df_kpi_summary = pd.read_sql(text(overall_kpi_sql), engine)
df_kpi_summary


In [ ]:
kpi = df_kpi_summary.iloc[0]

print('Overall KPI Summary')
print('-' * 40)
print(f"Total GMV   : {kpi['total_gmv']}")
print(f"Total Orders: {int(kpi['total_orders'])}")
print(f"Total Buyers: {int(kpi['total_buyers'])}")
print(f"Overall AOV : {kpi['overall_aov']}")


## Monthly KPI Trends

In [ ]:
monthly_kpi_sql = """
WITH delivered_orders AS (
    SELECT
        order_id,
        customer_unique_id,
        DATE_FORMAT(order_purchase_timestamp, '%Y-%m') AS order_month,
        gmv
    FROM order_level_dataset
    WHERE order_status = 'delivered'
)
SELECT
    order_month,
    ROUND(SUM(gmv), 2) AS monthly_gmv,
    COUNT(DISTINCT order_id) AS monthly_orders,
    COUNT(DISTINCT customer_unique_id) AS monthly_buyers,
    ROUND(SUM(gmv) / NULLIF(COUNT(DISTINCT order_id), 0), 2) AS monthly_aov
FROM delivered_orders
GROUP BY order_month
ORDER BY order_month;
"""

df_monthly_kpi = pd.read_sql(text(monthly_kpi_sql), engine)
df_monthly_kpi['order_month'] = pd.to_datetime(df_monthly_kpi['order_month'], format='%Y-%m')
df_monthly_kpi = df_monthly_kpi.sort_values('order_month')
df_monthly_kpi.head()


In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_monthly_kpi, x='order_month', y='monthly_gmv', marker='o')
plt.title('Monthly GMV Trend')
plt.xlabel('Order Month')
plt.ylabel('GMV')
plt.xticks(rotation=45)
plt.tight_layout()

output_path = DASHBOARD_DIR / 'monthly_gmv_trend.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

sns.lineplot(data=df_monthly_kpi, x='order_month', y='monthly_orders', marker='o', label='Orders', ax=ax)
sns.lineplot(data=df_monthly_kpi, x='order_month', y='monthly_buyers', marker='o', label='Buyers', ax=ax)

ax.set_title('Monthly Orders and Buyers Trend')
ax.set_xlabel('Order Month')
ax.set_ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()

output_path = DASHBOARD_DIR / 'monthly_orders_buyers_trend.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_monthly_kpi, x='order_month', y='monthly_aov', marker='o')
plt.title('Monthly AOV Trend')
plt.xlabel('Order Month')
plt.ylabel('AOV')
plt.xticks(rotation=45)
plt.tight_layout()

output_path = DASHBOARD_DIR / 'monthly_aov_trend.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


## Category Analysis

In [ ]:
top_category_gmv_sql = """
WITH delivered_order_ids AS (
    SELECT order_id
    FROM order_level_dataset
    WHERE order_status = 'delivered'
)
SELECT
    COALESCE(p.product_category_name, 'unknown') AS product_category_name,
    ROUND(SUM(oi.price), 2) AS category_gmv
FROM delivered_order_ids d
INNER JOIN order_items oi
    ON d.order_id = oi.order_id
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY COALESCE(p.product_category_name, 'unknown')
ORDER BY category_gmv DESC
LIMIT 10;
"""

df_top_category_gmv = pd.read_sql(text(top_category_gmv_sql), engine)
df_top_category_gmv


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=df_top_category_gmv, x='category_gmv', y='product_category_name')
plt.title('Top 10 Categories by GMV')
plt.xlabel('GMV')
plt.ylabel('Product Category')
plt.tight_layout()

output_path = DASHBOARD_DIR / 'top_categories_gmv.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


In [ ]:
top_category_orders_sql = """
WITH delivered_order_ids AS (
    SELECT order_id
    FROM order_level_dataset
    WHERE order_status = 'delivered'
)
SELECT
    COALESCE(p.product_category_name, 'unknown') AS product_category_name,
    COUNT(DISTINCT oi.order_id) AS category_order_count
FROM delivered_order_ids d
INNER JOIN order_items oi
    ON d.order_id = oi.order_id
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY COALESCE(p.product_category_name, 'unknown')
ORDER BY category_order_count DESC
LIMIT 10;
"""

df_top_category_orders = pd.read_sql(text(top_category_orders_sql), engine)
df_top_category_orders


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=df_top_category_orders, x='category_order_count', y='product_category_name')
plt.title('Top 10 Categories by Order Count')
plt.xlabel('Order Count')
plt.ylabel('Product Category')
plt.tight_layout()
plt.show()


## Review and Fulfillment Analysis

In [ ]:
review_distribution_sql = """
SELECT
    review_score,
    COUNT(*) AS review_count
FROM order_level_dataset
WHERE order_status = 'delivered'
  AND review_score IS NOT NULL
GROUP BY review_score
ORDER BY review_score;
"""

df_review_distribution = pd.read_sql(text(review_distribution_sql), engine)
df_review_distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df_review_distribution, x='review_score', y='review_count')
plt.title('Review Score Distribution')
plt.xlabel('Review Score')
plt.ylabel('Order Count')
plt.tight_layout()
plt.show()


In [ ]:
late_delivery_monthly_sql = """
SELECT
    DATE_FORMAT(order_purchase_timestamp, '%Y-%m') AS order_month,
    ROUND(AVG(is_late_delivery) * 100, 2) AS late_delivery_rate
FROM order_level_dataset
WHERE order_status = 'delivered'
  AND is_late_delivery IS NOT NULL
GROUP BY DATE_FORMAT(order_purchase_timestamp, '%Y-%m')
ORDER BY order_month;
"""

df_late_delivery = pd.read_sql(text(late_delivery_monthly_sql), engine)
df_late_delivery['order_month'] = pd.to_datetime(df_late_delivery['order_month'], format='%Y-%m')
df_late_delivery = df_late_delivery.sort_values('order_month')
df_late_delivery.head()


In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_late_delivery, x='order_month', y='late_delivery_rate', marker='o')
plt.title('Monthly Late Delivery Rate')
plt.xlabel('Order Month')
plt.ylabel('Late Delivery Rate (%)')
plt.xticks(rotation=45)
plt.tight_layout()

output_path = DASHBOARD_DIR / 'late_delivery_rate_trend.png'
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', output_path)


In [ ]:
delivery_review_sql = """
SELECT
    CASE
        WHEN is_late_delivery = 1 THEN 'Late Delivery'
        WHEN is_late_delivery = 0 THEN 'On-time Delivery'
        ELSE 'Unknown'
    END AS delivery_group,
    COUNT(*) AS order_count,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM order_level_dataset
WHERE order_status = 'delivered'
  AND review_score IS NOT NULL
  AND is_late_delivery IS NOT NULL
GROUP BY
    CASE
        WHEN is_late_delivery = 1 THEN 'Late Delivery'
        WHEN is_late_delivery = 0 THEN 'On-time Delivery'
        ELSE 'Unknown'
    END
ORDER BY avg_review_score;
"""

df_delivery_review = pd.read_sql(text(delivery_review_sql), engine)
df_delivery_review


In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=df_delivery_review, x='delivery_group', y='avg_review_score')
plt.title('Average Review Score: Late vs On-time Delivery')
plt.xlabel('Delivery Group')
plt.ylabel('Average Review Score')
plt.tight_layout()
plt.show()


## Key Business Insights

- Monthly GMV trend reflects overall business growth stability.
- Orders and Buyers trends help identify whether GMV growth is driven by scale or demand efficiency.
- Monthly AOV trend helps evaluate basket-size stability over time.
- Top categories by GMV reveal the main revenue contributors.
- Late delivery trend and review score comparison help explain customer experience risk.

## Recommended Actions

- Prioritize high-GMV categories in promotion and inventory planning.
- Track whether GMV growth relies more on buyer growth or stronger order value.
- Investigate periods with weak AOV performance.
- Reduce late delivery rate to improve ratings and customer trust.
- Combine category and fulfillment insights for more actionable business decisions.